<a href="https://colab.research.google.com/github/varshayinithangaraju30-lgtm/customer-segmentation-kmeans/blob/main/dashboard_ui.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

URL = "https://raw.githubusercontent.com/varshayinithangaraju30-lgtm/customer-segmentation-kmeans/main/Mall_Customers.csv"
FEATURES = ["Age", "Annual Income (k$)", "Spending Score (1-100)"]

df = pd.read_csv(URL).drop_duplicates().dropna(subset=FEATURES)
X = StandardScaler().fit_transform(df[FEATURES])


def name_cluster(row):
    age = "Young" if row["Age"] < 35 else "Older"
    s = row["Spending Score (1-100)"]
    spend = "High" if s >= 60 else ("Medium" if s >= 35 else "Low")
    return f"{age} {spend} Spenders"


def build_dashboard(k, x_feature, y_feature):
    data = df.copy()
    labels = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=42).fit_predict(X)
    data["Cluster ID"] = labels
    means = data.groupby("Cluster ID")[FEATURES].mean()
    names = {cid: f"{cid}: {name_cluster(row)}" for cid, row in means.iterrows()}
    data["Segment"] = data["Cluster ID"].map(names)

    fig = px.scatter(
        data, x=x_feature, y=y_feature, color="Segment",
        hover_name="Segment",
        hover_data={"CustomerID": True, "Gender": True, "Age": True,
                    "Annual Income (k$)": True, "Spending Score (1-100)": True,
                    "Segment": False},
        title=f"Customer Segments (K = {k})",
        template="plotly_white",
    )
    fig.update_traces(marker=dict(size=14, line=dict(width=1, color="white")))
    fig.update_layout(legend_title_text="Segment", height=480)

    cards = ""
    for cid, row in means.iterrows():
        size = int((data["Cluster ID"] == cid).sum())
        cards += f"""
        <div style="flex:1;min-width:180px;margin:6px;padding:12px;border-radius:10px;
                    background:#f4f6fb;border:1px solid #dfe3ee;color:#222;font-family:sans-serif">
          <b>{names[cid]}</b><br>
          Customers: {size}<br>
          Avg age: {row['Age']:.0f}<br>
          Avg income: {row['Annual Income (k$)']:.0f}k$<br>
          Avg spending: {row['Spending Score (1-100)']:.0f}
        </div>"""
    return fig, f"<div style='display:flex;flex-wrap:wrap'>{cards}</div>"


k_slider = widgets.IntSlider(value=3, min=2, max=6, description="Clusters (K)")
x_drop = widgets.Dropdown(options=FEATURES, value=FEATURES[1], description="X axis")
y_drop = widgets.Dropdown(options=FEATURES, value=FEATURES[2], description="Y axis")
out = widgets.Output()


def update(_=None):
    fig, cards = build_dashboard(k_slider.value, x_drop.value, y_drop.value)
    out.clear_output(wait=True)
    with out:
        display(HTML("<h3 style='font-family:sans-serif'>Customer Segmentation Dashboard</h3>"))
        display(HTML(cards))
        display(widgets.HTML(fig.to_html(include_plotlyjs="cdn", full_html=False)))


for w in (k_slider, x_drop, y_drop):
    w.observe(update, names="value")

display(widgets.VBox([k_slider, x_drop, y_drop, out]))
update()

In [4]:
fig, cards = build_dashboard(3, "Annual Income (k$)", "Spending Score (1-100)")
fig.show()